# Stage 4: gradient-boosting models (LightGBM and XGBoost)

This notebook only **shows** results. The work is done by two scripts:

- `python -m src.tune_gbm --horizon <h> --trials 20` : random search, validation only;
- `python -m src.train_gbm` : final refit on train + validation, one test run, saved models.

**Why tree ensembles?** The features are a table (lags, weather, sun position, calendar). Trees handle
missing values, need no scaling, and learn interactions (for example: irradiance matters differently
at noon and at sunset) without us writing them by hand.

**How was the test set kept clean?** Every setting is fitted on *train* and scored on *validation*
(early stopping also on validation). The best setting and its number of trees are fixed. Only then is
the model refit on train + validation and the 2023 test year scored **once**.

In [ ]:
import pandas as pd
from IPython.display import Image

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)
TABLES = "../results/tables/"
FIGURES = "../results/figures/"

## 1. Tuning log
Every setting that was tried. The lowest validation RMSE per model and horizon wins.

In [ ]:
log = pd.read_csv(TABLES + "04_tuning_log.csv")
log.groupby(["horizon", "model"])["val_rmse"].agg(["count", "min", "median", "max"]).round(5)

In [ ]:
best = log.loc[log.groupby(["horizon", "model"])["val_rmse"].idxmin()]
best[["horizon", "model", "trial", "val_rmse", "n_trees", "params"]]

## 2. Test results next to the baselines
Daytime, seen stations, 2023. `skill` = 1 - RMSE / RMSE of the benchmark persistence.

In [ ]:
baselines = pd.read_csv(TABLES + "03_baselines.csv")
gbm = pd.read_csv(TABLES + "04_gbm_results.csv")
both = pd.concat([baselines, gbm])

In [ ]:
test = both[(both["split"] == "test") & (both["hours"] == "daytime")]
columns = ["horizon", "model", "nmae_pct", "nmae_pct_low", "nmae_pct_high", "skill", "skill_low", "skill_high", "mean_station_skill"]
test[columns].sort_values(["horizon", "nmae_pct"]).round(3)

In [ ]:
Image(FIGURES + "04_gbm_skill.png")

In [ ]:
Image(FIGURES + "04_gbm_nmae.png")

## 3. All hours
Night targets are 0 and night forecasts are 0 by rule, so all-hours errors are smaller. Skill is almost the same.

In [ ]:
all_hours = both[(both["split"] == "test") & (both["hours"] == "all")]
all_hours[["horizon", "model", "nmae_pct", "skill"]].sort_values(["horizon", "nmae_pct"]).round(3)

## 4. Validation vs test
Validation is one dry season (Oct-Dec 2022); the test year has all seasons, so the errors differ.

In [ ]:
day = gbm[gbm["hours"] == "daytime"]
day.pivot(index=["horizon", "model"], columns="split", values=["nmae_pct", "skill"]).round(3)

## 5. Per season (test, daytime)

In [ ]:
season = pd.read_csv(TABLES + "04_gbm_per_season.csv")
season.pivot(index=["horizon", "model"], columns="season", values="nmae_pct").round(2)

In [ ]:
season.pivot(index=["horizon", "model"], columns="season", values="skill").round(3)

## 6. Per station (test, daytime)
Spread of the skill over the stations: is the model better than persistence everywhere?

In [ ]:
station = pd.read_csv(TABLES + "04_gbm_per_station.csv")
station.groupby(["horizon", "model"])["skill"].describe().round(3)

In [ ]:
worst = station.sort_values("skill").groupby(["horizon", "model"]).head(3)
worst[["horizon", "model", "station", "nmae_pct", "skill"]].round(3)